In [43]:
!pip install fastapi uvicorn python-multipart transformers torch Pillow omegaconf

In [44]:
!pip install fastapi uvicorn nest-asyncio omegaconf

In [45]:
yaml_config = """
                model_path: "google/vit-base-patch16-224"
            """
with open("config.yaml", "w", encoding="utf-8") as f:
  f.write(yaml_config)

In [46]:
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import requests

url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)

processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

inputs = processor(images=image, return_tensors="pt")
outputs = model(**inputs)
logits = outputs.logits
# model predicts one of the 1000 ImageNet classes
predicted_class_idx = logits.argmax(-1).item()
print("Predicted class:", model.config.id2label[predicted_class_idx])


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Predicted class: Egyptian cat


#Build Model

In [47]:
import torch
from transformers import ViTImageProcessor, ViTForImageClassification
from omegaconf import OmegaConf
from PIL import Image

class ImageClassificationModel:
    def __init__(self, config_path: str):
        # Tải cấu hình từ file YAML
        self.config = OmegaConf.load(config_path)

        # Tải bộ tiền xử lý ảnh và trọng số mạng nơ-ron từ Hugging Face
        self.processor = ViTImageProcessor.from_pretrained(self.config.model_path)
        self.model = ViTForImageClassification.from_pretrained(self.config.model_path)

        # Tối ưu hóa phần cứng: Cấp phát vào GPU nếu có, ngược lại dùng CPU
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

        # Thiết lập mô hình ở trạng thái suy luận (vô hiệu hóa Dropout/BatchNorm training)
        self.model.eval()

    def predict(self, image: Image.Image) -> dict:
        # Tiền xử lý (đổi kích thước thành 224x224, chuẩn hóa pixel) và chuyển thành Tensor
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)

        # Môi trường tính toán nghiêm ngặt: Tắt cơ chế đạo hàm (Gradient Calculation)
        with torch.inference_mode():
            outputs = self.model(**inputs)

        # Trích xuất giá trị raw (logits) và đưa qua hàm Softmax để tính phân phối xác suất
        logits = outputs.logits
        probabilities = torch.nn.functional.softmax(logits, dim=-1)

        # Định vị nơ-ron có giá trị kích hoạt lớn nhất (chỉ số lớp dự đoán)
        predicted_class_idx = logits.argmax(-1).item()
        confidence_score = probabilities[0, predicted_class_idx].item()

        # Ánh xạ chỉ số toán học thành nhãn ngôn ngữ tự nhiên
        predicted_label = self.model.config.id2label[predicted_class_idx]

        return {
            "predicted_class": predicted_label,
            "confidence": round(confidence_score, 4) # Làm tròn 4 chữ số thập phân
        }

In [48]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
from PIL import Image, UnidentifiedImageError
import io


# Khởi tạo ứng dụng FastAPI
app = FastAPI(
    title="Computer Vision API",
    description="Hệ thống phân loại hình ảnh sử dụng mô hình Vision Transformer (ViT)."
)

# Khởi tạo thực thể (instance) mô hình duy nhất trong bộ nhớ (Singleton Pattern)
try:
    classifier = ImageClassificationModel("config.yaml")
except Exception as e:
    print(f"Lỗi nghiêm trọng khi khởi tạo mạng nơ-ron: {e}")
    classifier = None

@app.get("/")
async def root():
    """ Trả về thông tin giới thiệu ngắn gọn về hệ thống hoặc mô tả chức năng của API. """
    return {
        "system": "Computer Vision API",
        "framework": "FastAPI & PyTorch",
        "endpoints": ["/health", "/predict"]
    }

@app.get("/health")
async def health_check():
    """ Kiểm tra trạng thái hoạt động của hệ thống. """
    if classifier is None:
        raise HTTPException(status_code=503, detail="Mô hình chưa được nạp vào bộ nhớ.")
    return {"status": "ok", "model_loaded": True}

@app.post("/predict")
async def predict_image(file: UploadFile = File(...)):
    """
    Nhận dữ liệu đầu vào từ người dùng, gọi mô hình Hugging Face, và trả kết quả dưới dạng JSON.
    """
    # Rào chắn bảo mật 1: Kiểm tra MIME type ở mức cơ bản
    if not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="Định dạng tệp từ chối. Vui lòng tải lên chuẩn hình ảnh (JPEG/PNG).")

    try:
        # Nạp luồng byte vật lý từ giao thức mạng
        image_bytes = await file.read()
        # Chuyển đổi thành ma trận điểm ảnh RGB
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    except UnidentifiedImageError:
        # Rào chắn bảo mật 2: Xử lý hợp lý các trường hợp lỗi như sai định dạng thực tế
        raise HTTPException(status_code=400, detail="Cấu trúc tệp bị hỏng hoặc không phải là hình ảnh hợp lệ.")
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Lỗi I/O nội bộ: {str(e)}")

    try:
        # Kích hoạt quá trình suy luận
        result = classifier.predict(image)
        # Kết quả trả về phải rõ ràng, có cấu trúc và ở định dạng JSON
        return JSONResponse(content=result)
    except Exception as e:
        # Bắt lỗi tính toán Tensor (nếu có)
        raise HTTPException(status_code=500, detail=f"Lỗi trong quá trình suy luận toán học: {str(e)}")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

In [50]:
import threading
import uvicorn
import nest_asyncio

# Cho phép chạy lồng vòng lặp sự kiện trong môi trường Notebook
try:
    nest_asyncio.apply()
except:
    pass

def run_server():
    # Khởi chạy server tại localhost cổng 8000
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Chạy server ở luồng phụ (background) để không khóa luồng chính
# daemon=True giúp thread tự tắt khi bạn đóng chương trình chính
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("Máy chủ đang chạy tại: http://127.0.0.1:8000")

Máy chủ đang chạy tại: http://127.0.0.1:8000


In [58]:
import requests
import sys

# Khai báo địa chỉ đích
API_URL = "http://127.0.0.1:8000/predict"

def verify_inference(image_path: str):
    print(f"[*] Đang đóng gói và truyền tải tệp: {image_path}...")
    try:
        # Mở luồng đọc nhị phân (Read Binary)
        with open(image_path, "rb") as file_stream:
            # Tuân thủ giao thức multipart/form-data
            files = {"file": (image_path, file_stream, "image/jpeg")}

            # Gửi HTTP POST request
            response = requests.post(API_URL, files=files)

        if response.status_code == 200:
            print("[+] Suy luận thành công. Dữ liệu phản hồi:")
            print(response.json())
        else:
            print(f"[-] Hệ thống từ chối yêu cầu. Mã lỗi: {response.status_code}")
            print(f"[-] Chi tiết: {response.text}")

    except FileNotFoundError:
        print(f"[!] Ngoại lệ: Không tìm thấy tệp {image_path} trên đĩa cứng.")
    except Exception as e:
        print(f"[!] Lỗi mạng hoặc I/O: {e}")

if __name__ == "__main__":
    # Thay đổi đường dẫn này thành một bức ảnh thực tế (ví dụ: ảnh con chó, con mèo) trên máy của bạn
    sample_image = "/content/hcmus.webp"
    verify_inference(sample_image)

[*] Đang đóng gói và truyền tải tệp: /content/hcmus.webp...
INFO:     127.0.0.1:45482 - "POST /predict HTTP/1.1" 200 OK
[+] Suy luận thành công. Dữ liệu phản hồi:
{'predicted_class': 'palace', 'confidence': 0.4171}
